# LAB | Abstractive Question Answering

Abstractive question-answering focuses on the generation of multi-sentence answers to open-ended questions. It usually works by searching massive document stores for relevant information and then using this information to synthetically generate answers. This notebook demonstrates how Pinecone helps you build an abstractive question-answering system. We need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A generator model to generate answers

# Install Dependencies

In [1]:
!pip uninstall torchcodec -y -q
!pip install torchvision==0.26.0+cu130 \
    --index-url https://download.pytorch.org/whl/cu130 \
    --force-reinstall -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 114.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 163.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 153.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 204.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 107.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 258.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 129.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 189.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 247.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 203.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 91.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.9/145.9 MB 139.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import torch, torchvision
print(f"PyTorch:     {torch.__version__}")      # 2.11.0+cu130
print(f"torchvision: {torchvision.__version__}") # 0.26.0+cu130
print(f"CUDA:        {torch.cuda.is_available()}") # True

PyTorch:     2.11.0+cu130
torchvision: 0.26.0+cu130
CUDA:        True


In [2]:
!pip install -qU datasets pinecone-client==3.1.0 \
    sentence-transformers transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.0/211.0 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.7/588.7 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 157.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 58.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.


In [3]:
#!pip install -U langchain langchain-core langchain-classic langchain-pinecone langchain-huggingface datasets pinecone-client sentence-transformers torch

# Load and Prepare Dataset

Our source data will be taken from the Wiki Snippets dataset, which contains over 17 million passages from Wikipedia. But, since indexing the entire dataset may take some time, we will only utilize 50,000 passages in this demo that include "History" in the "section title" column. If you want, you may utilize the complete dataset. Pinecone vector database can effortlessly manage millions of documents for you.

In [4]:

from datasets import load_dataset

# ✅ Modern parquet-based dataset, no scripts needed
wiki_data = load_dataset(
    'wikimedia/wikipedia',
    '20231101.en',
    split='train',
    streaming=True
).shuffle(seed=960, buffer_size=10_000)

# verify
sample = next(iter(wiki_data))
print(sample.keys())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

dict_keys(['id', 'url', 'title', 'text'])


We are loading the dataset in the streaming mode so that we don't have to wait for the whole dataset to download (which is over 9GB). Instead, we iteratively download records one at a time.

In [5]:
# show the contents of a single document in the dataset
next(iter(wiki_data))

{'id': '2117822',
 'url': 'https://en.wikipedia.org/wiki/57th%20United%20States%20Congress',
 'title': '57th United States Congress',
 'text': 'The 57th United States Congress was a meeting of the legislative branch of the United States federal government, composed of the United States Senate and the United States House of Representatives. It met in Washington, DC from March 4, 1901, to March 4, 1903, during the final six months of William McKinley\'s presidency, and the first year and a half of the first administration of his successor, Theodore Roosevelt. The apportionment of seats in the House of Representatives was based on the 1890 United States census. Both chambers had a Republican majority.\n\nMajor events\n\n September 6, 1901: Leon Czolgosz shot President William McKinley at the Pan-American Exposition in Buffalo, New York\n September 14, 1901: President William McKinley died. Vice President Theodore Roosevelt became President of the United States\n October 16, 1901: Presiden

In [6]:
# filter only documents with History as title
history = wiki_data.filter(
    lambda d: 'history' in d['title'].lower()
)



Let's iterate through the dataset and apply our filter to select the 50,000 historical passages. We will extract `article_title`, `section_title` and `passage_text` from each document.

In [7]:

# extract correct fields
from tqdm.auto import tqdm

total_doc_count = 10000
counter = 0
docs = []

for d in tqdm(history, total=total_doc_count):
    # split long text into 500 char chunks as passages
    text = d['text']
    chunks = [text[i:i+500] for i in range(0, min(len(text), 2000), 500)]
    for chunk in chunks:
        if chunk.strip():
            docs.append({
                'article_title': d['title'],
                'section_title': 'History',
                'passage_text':  chunk
            })
    counter += 1
    if counter >= total_doc_count or len(docs) >= total_doc_count:
        break

print(f"Total passages collected: {len(docs)}")

  0%|          | 0/10000 [00:00<?, ?it/s]

Total passages collected: 10003


In [8]:
import pandas as pd

# create a pandas dataframe with the documents we extracted
df = pd.DataFrame(docs)
df.head()

,article_title,section_title,passage_text
0,We Are History,History,We Are History is a British comedy series broa...
1,We Are History,History,"an IKEA store. In another, he recreated the Sp..."
2,We Are History,History,rench visitors who overstayed their welcome an...
3,We Are History,History,"ada!""\nOriginal Air Date: 14 April 2000\n\nSea..."
4,Sam Noble Oklahoma Museum of Natural History,History,The Sam Noble Oklahoma Museum of Natural Histo...


# Initialize Pinecone Index

The Pinecone index stores vector representations of our historical passages which we can retrieve later using another vector (query vector). To build our vector index, we must first establish a connection with Pinecone. For this, we need an API from Pinecone. You can get one for free from [here](https://app.pinecone.io/), and after that, we initialize the connection as follows:

In [9]:
from google.colab import userdata
import os
from pinecone import Pinecone
from pinecone import ServerlessSpec

from google.colab import userdata
pinecone_api_key = userdata.get('PINECONE_API_KEY')




spec = ServerlessSpec(
    cloud="aws", region="us-east-1"
)

# connect to pinecone environment
pc = Pinecone(api_key=pinecone_api_key)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

Now we create a new index. We will name it "abstractive-question-answering" — you can name it anything we want. We specify the metric type as "cosine" and dimension as 768 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 768-dimension vectors.

In [11]:
index_name = "abstractive-question-answering"

import time

if index_name not in pc.list_indexes().names():
    pc.create_index(
        index_name,
        dimension=768,
        metric='cosine',
        spec=spec
    )

    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

index = pc.Index(index_name)
index.describe_index_stats()

{'dimension': 768,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}

# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all historical passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will create embeddings such that the questions and passages that hold the answers to our queries are close to one another in the vector space. We will use a SentenceTransformer model based on Microsoft's MPNet as our retriever. This model performs quite well for comparing the similarity between queries and documents. We can use Cosine Similarity to compute the similarity between query and context vectors generated by this model (Pinecone automatically does this for us).

In [12]:
import torch
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using: {device}")

# ✅ FIXED: load correct model
retriever = SentenceTransformer(
    'flax-sentence-embeddings/all_datasets_v3_mpnet-base',
    device=device
)
retriever

Using: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'MPNetModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, section title, passage text, etc.

In [13]:
# we will use batches of 64
batch_size = 64

for i in tqdm(range(0, len(df), batch_size)):
    # find end of batch
    i_end = min(i + batch_size, len(df))
    # extract batch
    batch = df.iloc[i:i_end]
    # generate embeddings for batch
    embeds = retriever.encode(batch['passage_text'].tolist()).tolist()
    # create metadata
    meta = batch[['article_title', 'section_title', 'passage_text']].to_dict(orient='records')
    # create unique IDs
    ids = [str(x) for x in range(i, i_end)]
    # upsert to pinecone
    index.upsert(vectors=zip(ids, embeds, meta))

# check that we have all vectors in index
index.describe_index_stats()

  0%|          | 0/157 [00:00<?, ?it/s]

{'dimension': 768,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 10003}},
 'total_vector_count': 10003}

# Initialize Generator

We will use ELI5 BART for the generator which is a Sequence-To-Sequence model trained using the ‘Explain Like I’m 5’ (ELI5) dataset. Sequence-To-Sequence models can take a text sequence as input and produce a different text sequence as output.

The input to the ELI5 BART model is a single string which is a concatenation of the query and the relevant documents providing the context for the answer. The documents are separated by a special token &lt;P>, so the input string will look as follows:

>question: What is a sonic boom? context: &lt;P> A sonic boom is a sound associated with shock waves created when an object travels through the air faster than the speed of sound. &lt;P> Sonic booms generate enormous amounts of sound energy, sounding similar to an explosion or a thunderclap to the human ear. &lt;P> Sonic booms due to large supersonic aircraft can be particularly loud and startling, tend to awaken people, and may cause minor damage to some structures. This led to prohibition of routine supersonic flight overland.

More detail on how the ELI5 dataset was built is available [here](https://arxiv.org/abs/1907.09190) and how ELI5 BART model was trained is available [here](https://yjernite.github.io/lfqa.html).

Let's initialize the BART model using transformers.

In [14]:
from transformers import BartTokenizer, BartForConditionalGeneration

tokenizer = BartTokenizer.from_pretrained('vblagoje/bart_lfqa')
generator = BartForConditionalGeneration.from_pretrained(
    'vblagoje/bart_lfqa'
).to(device)

tokenizer_config.json:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

All the components of our abstract QA system are complete and ready to be queried. But first, let's write some helper functions to retrieve context passages from Pinecone index and to format the query in the way the generator expects the input.

In [15]:
def query_pinecone(query, top_k):
    # generate embeddings for the query
    xq = retriever.encode([query]).tolist()[0]
    # search pinecone index for context passage with the answer
    xc = index.query(
        vector=xq,
        top_k=top_k,
        include_metadata=True
    )
    return xc

In [16]:
def format_query(query, context):
    # extract passage_text from Pinecone search result and add the <P> tag
    context = [f"<P> {m['metadata']['passage_text']}" for m in context]
    # concatinate all context passages
    context = ' '.join(context)
    # contcatinate the query and context passages
    query = f"question: {query} context: {context}"
    return query

Let's test the helper functions. We will query the Pinecone index function we created earlier with the `query_pinecone` to get context passages and pass them to the `format_query` function.

In [17]:
query = "when was the first electric power system built?"
result = query_pinecone(query, top_k=1)
result

{'matches': [{'id': '7957',
              'metadata': {'article_title': 'History of electricity supply in '
                                            'Brisbane',
                           'passage_text': '\n'
                                           '\n'
                                           'The first practical use of '
                                           'electricity was for lighting in '
                                           'the Government Printing Office in '
                                           'George Street in April 1883. In '
                                           '1886, the Roma Street Railway '
                                           'Yards were using arc lights, and '
                                           'in the same year, an underground '
                                           'cable connected the Parliament '
                                           'House from the Printing Office, '
                                          

In [18]:
from pprint import pprint

In [19]:
# format the query in the form generator expects the input
query = format_query(query, result['matches'])
pprint(query)

('question: when was the first electric power system built? context: <P> \n'
 '\n'
 'The first practical use of electricity was for lighting in the Government '
 'Printing Office in George Street in April 1883. In 1886, the Roma Street '
 'Railway Yards were using arc lights, and in the same year, an underground '
 'cable connected the Parliament House from the Printing Office, the first of '
 'any Parliament House in Australia. The supervision of the laying of cable '
 'was done by E.C. Barton, who formed a company with C.F. White and in 1888 '
 'built a power house in Edison Lane behind the General')


The output looks great. Now let's write a function to generate answers.

In [20]:
def generate_answer(query):
    # tokenize the query to get input_ids
    inputs = tokenizer([query], max_length=1024, return_tensors="pt").to(device)
    # use generator to predict output ids
    ids = generator.generate(inputs["input_ids"], num_beams=2, min_length=20, max_length=40)
    # use tokenizer to decode the output ids
    answer = tokenizer.batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return pprint(answer)

In [21]:
generate_answer(query)

('The first electric power system was built in the United States in 1883. The '
 'first practical use of electricity was for lighting in the Government '
 'Printing Office in George Street in April 1883. In 1886')


As we can see, the generator used the provided context to answer our question. Let's run some more queries.

In [22]:
query = "How was the first wireless message sent?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

('The first wireless message was sent by a telegraph. The first telegraph was '
 'built in London in 1837. The first telegraph was built in 1839. The first '
 'telegraph was built')


To confirm that this answer is correct, we can check the contexts used to generate the answer.

In [23]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

ot come into common use until several years later) was sketchy, with magazines such as the November, 1901 issue of Amateur Work showing how to build a simple system based on Hertz' early experiments. Magazines show a continued progress by amateurs including a 1904 story on two Boston, Massachusetts 8th graders constructing a transmitter and receiver with a range of eight miles and a 1906 story about two Rhode Island teenagers building a wireless station in a chicken coop. In the US the first com
---
ame into being after radio waves (proved to exist by Heinrich Rudolf Hertz in 1888) were adapted into a communication system in the 1890s by the Italian inventor Guglielmo Marconi. In the late 19th century there had been amateur wired telegraphers setting up their own interconnected telegraphic systems. Following Marconi's success many people began experimenting with this new form of "wireless telegraphy". Information on "Hertzian wave" based wireless telegraphy systems (the name "radio" wo

In this case, the answer looks correct. If we ask a question and no relevant contexts are retrieved, the generator will typically return nonsensical or false answers, like with this question about COVID-19:

In [24]:
query = "where did COVID-19 originate?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

('COVID-19 is a new strain of HIV that has only been around for a few years. '
 'It is thought to have originated in the Congo in the 1920s. The current '
 'outbreak is caused')


In [25]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

es of potatoes and rinderpest of cattle had devastating consequences.

Smallpox and measles viruses are among the oldest that infect humans. Having evolved from viruses that infected other animals, they first appeared in humans in  Europe and  North Africa thousands of years ago. The viruses were later carried to the New World by Europeans during the time of the Spanish Conquests, but the indigenous people had no natural resistance to the viruses and millions of them died during epidemics. Influ
---
The social history of viruses describes the influence of viruses and viral infections on human history. Epidemics caused by viruses began when human behaviour changed during the Neolithic period, around 12,000 years ago, when humans developed more densely populated agricultural communities. This allowed viruses to spread rapidly and subsequently to become endemic. Viruses of plants and livestock also increased, and as humans became dependent on agriculture and farming, diseases such as poty

Let’s finish with a final few questions.

In [26]:
query = "what was the war of currents?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

('The war of currents is a term used to refer to a series of events that '
 'occurred in the early 20th century. The most famous of these events was the '
 'Battle of Tsushima, which was')


In [27]:
query = "who was the first person on the moon?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

('The first person to go to the moon was probably Giovanni Cassini, who landed '
 'on the moon in 1969. He landed on the moon in 1969.')


In [28]:
query = "what was NASAs most expensive project?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

("I'm not sure if this counts as a project, but I think it's worth noting that "
 "the National Science Foundation's budget for the year was $2.5 billion.")


As we can see, the model can generate some decent answers.

#### Add a few more questions

In [29]:
query = "Who invented television?"
context = query_pinecone(query, top_k=10)
query = format_query(query, context["matches"])
generate_answer(query)

("I'm not sure if this is the right subreddit to ask this question, but I'll "
 "give it a shot anyway. I'm not a historian, but I'm a computer science "
 'major, so')


In [30]:
query = "Who designed the Sydney Opera house?"
context = query_pinecone(query, top_k=10)
query = format_query(query, context["matches"])
generate_answer(query)

('The Sydney Opera House was designed by the Irish firm Ireland & Associates. '
 'The building was built in the 1960s, and was completed in the 1980s. The '
 'building was designed by the Irish firm')


In [31]:
query = "How is Joghurt produced?"
context = query_pinecone(query, top_k=10)
query = format_query(query, context["matches"])
generate_answer(query)

("I'm not sure if this is what you're looking for, but here's how it's done: "
 '1. Get a bunch of grapes and let them ferment for a while. 2. Put')
